# einops-rearrange — ex6: multi-head attention split (with shape-pipeline debug)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-rearrange`. Running the final beacon cell reports progress against the `Einops: Rearrange` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Rearrange` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-rearrange`** (exercise 6). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-rearrange"
DD_SUBTOPIC = "Einops: Rearrange"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.rearrange — quick refresher

`rearrange(tensor, pattern, **axes_lengths)` is one operator with three jobs: **reorder** axes (`'h w -> w h'`), **compose** them (`'h w c -> (h w) c'`), and **decompose** them (`'(b1 b2) c -> b1 b2 c'`, with `b1=` or `b2=`). Every identifier on the right must appear on the left and vice versa.

The exercises below build on that: each one runs `rearrange` inside a small pipeline where you have to *see* what the layout did — by plotting it, by printing the shape at each step, or by combining 2–3 patterns into a single ML-adjacent transformation.

### Exercise 6 — multi-head attention split (with shape-pipeline debug)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Use rearrange to split a packed (b, s, h*d) tensor into the (b, h, s, d) layout expected by scaled-dot-product attention, and verify the intermediate shapes by printing them.
> Keywords: multi-head, attention, axis-decomposition, shape-debug
> ```

**KCs targeted:** `rearrange-axis-decomposition`, `rearrange-axis-swap`

In multi-head attention, the projected Q/K/V tensors arrive as `(batch, seq_len, n_heads * head_dim)`. Before the attention matmul, you need to split the last axis into `n_heads` and `head_dim`, then move `n_heads` next to `batch` so attention scores can be computed per-head in a single batched matmul.

Implement `ex6_split_heads(x, n_heads)` that takes `x` of shape `(b, s, h*d)` and returns `(b, h, s, d)` using **one** `rearrange` call. Before returning, **print** the input shape, the intermediate shape if you had split first then moved (you can compute it from the input shape — you don't need a second rearrange), and the output shape. The print is the point: this is the kind of multi-step transformation you have to feel in your fingers when debugging attention bugs.

Hint: the pattern `'b s (h d) -> b h s d'` does both jobs in one go when you pass `h=n_heads`.

In [ ]:
def ex6_split_heads(x: Tensor, n_heads: int) -> Tensor:
    b, s, hd = x.shape
    d = hd // n_heads
    print(f'input         : {tuple(x.shape)}  (b, s, h*d)')
    print(f'if split only : ({b}, {s}, {n_heads}, {d})  (b, s, h, d)')
    print(f'after swap    : ({b}, {n_heads}, {s}, {d})  (b, h, s, d)')
    return rearrange(x, 'b s (h d) -> b h s d', h=n_heads)


<details><summary>Solution</summary>

```python
def ex6_split_heads(x: Tensor, n_heads: int) -> Tensor:
    b, s, hd = x.shape
    d = hd // n_heads
    print(f'input         : {tuple(x.shape)}  (b, s, h*d)')
    print(f'if split only : ({b}, {s}, {n_heads}, {d})  (b, s, h, d)')
    print(f'after swap    : ({b}, {n_heads}, {s}, {d})  (b, h, s, d)')
    return rearrange(x, 'b s (h d) -> b h s d', h=n_heads)
```

**Why one rearrange instead of two?** `'b s (h d) -> b h s d'` fuses the *decompose* (`(h d)` on the left, `h d` on the right) with the *reorder* (the `h` axis moves between `s` and `d`). Doing it in one call avoids an extra intermediate tensor — and, more importantly, makes the *intent* legible: any reader sees the head-split AND the swap in one line.

**Why print the pipeline?** Attention bugs often come from heads and seq_len being swapped, or from `head_dim` being computed wrong. Calling out the shape at each conceptual step (split → swap) makes the bug visible without a debugger.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex6'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex6',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()